In [101]:
FILE_ID = 405

In [102]:
files_dict = {
    406: {
        "main_padding": "./php/406/406_whole.php",
        "buggy_content": "./php/406/406_smallestBuggy.php",
        "buggy_line": "$selectedIds = explode(',', $selectedIds);",
        "split_string": "// -x-",
        "CWE_ID": "CWE-89"
    },
    408: {
        'main_padding': './php/408/408_whole.php',
        "additional_padding": ["./php/408/additional_padding.php"],
        "buggy_content": "./php/408/408_smallestBuggy.php",
        "buggy_line": "'SELECT * FROM ' . static::table_name() . ' WHERE ' . $property .  ' = \'' . $value . '\' LIMIT 0,1'",
        "split_string": "// -x-",
        "CWE_ID": "CWE-89"
    },
    405: {
        'main_padding': './php/405/405_whole.php',
        "additional_padding": ["./php/405/additional_padding.php"],
        "buggy_content": "./php/405/405_smallestBuggy.php",
        "buggy_line": '''$where = "WHERE group_ID = {$group_id}";''',
        "split_string": "// -x-",
        "CWE_ID": "CWE-89"
    }
}

In [77]:
def relaxed_knapsack(items, capacity, tolerance=0):
    # Dynamic programming table
    dp = [[0] * (capacity + tolerance + 1) for _ in range(len(items) + 1)]
    for i in range(1, len(items) + 1):
        item_size = items[i - 1]['size']
        for j in range(capacity + tolerance + 1):
            if item_size <= j:
                dp[i][j] = max(dp[i - 1][j], dp[i - 1][j - item_size] + item_size)
            else:
                dp[i][j] = dp[i - 1][j]
    # Find the best value close to capacity
    best_value = max(dp[len(items)][capacity:capacity + tolerance + 1])
    # Backtrack to find selected items
    result = []
    w = dp[len(items)].index(best_value)
    for i in range(len(items), 0, -1):
        if dp[i][w] != dp[i - 1][w]:
            result.append(items[i - 1])
            w -= items[i - 1]['size']
    return result

def parse_file(filename, split_string):
    snippets = []
    inside_snippet = False
    snippet_content = []

    with open(filename, 'r') as file:
        for line in file:
            # print(line)
            if split_string in line:
                # print("this line has split string")
                if inside_snippet:
                    # End of snippet
                    snippet = ''.join(snippet_content).strip()
                    snippets.append({
                        'size': len(snippet),
                        'snippet': snippet
                    })
                    snippet_content = []
                    inside_snippet = True
                else:
                    inside_snippet = not inside_snippet
            elif inside_snippet:
                snippet_content.append(line)
    
    return snippets

def get_padding_content(file_id, filename):
    file = files_dict[file_id]
    return parse_file(filename, file['split_string'])

In [96]:
def make_file_with_padding(file_id, target_chars, target_bug_position):
    # assert that bug position is not larger than target chars
    assert target_bug_position < target_chars
    # lets assume that we allow only mod 500 for both inputs
    assert target_bug_position % 500 == 0
    assert target_chars % 500 == 0

    with open(files_dict[file_id]['buggy_content'], 'r') as file:
        content = file.read()

    snippets = get_padding_content(file_id, files_dict[file_id]['main_padding'])
    buggy_chars = len(content) - len("{prepend_content}{append_content}")
    
    # find out how many 
    prepend_chars = target_bug_position
    append_chars = target_chars - prepend_chars - buggy_chars

    # count size of all snippets
    total_count = sum(x['size'] for x in snippets)

    print(len(snippets))
    if(total_count < prepend_chars + append_chars):
        for file in files_dict[file_id]['additional_padding']:
            snippets.extend(get_padding_content(file_id,file ))
    print(len(snippets))

    # generate snippets
    prepend_snippets = relaxed_knapsack(snippets, prepend_chars, 50) if prepend_chars > 50 else {}
    append_snippets = relaxed_knapsack(snippets, append_chars, 50) if append_chars > 50 else {}

    # generate the final content
    prepend_content = '\n'.join([snippet['snippet'] for snippet in prepend_snippets])
    append_content = '\n'.join([snippet['snippet'] for snippet in append_snippets])

    # insert into content (replace prepend_content and append_content)
    content = content.replace('{prepend_content}', prepend_content, 1)
    content = content.replace('{append_content}', append_content, 1)
    print(f"Prepend chars: {prepend_chars}")
    print(f"Append chars: {append_chars}")
    print(f"buggy length: {buggy_chars}")
    print(f"Prepend snippets: {len(prepend_content)}")
    print(f"Append snippets: {len(append_content)}")


    return content


In [97]:
get_padding_content(408, files_dict[408]['additional_padding'][0])

[{'size': 365,
  'snippet': '$correct_php_version = version_compare( phpversion(), "5.3", ">=" );\n\nif ( ! $correct_php_version ) {\n\tprintf( __( \'Podlove Subscribe Button Plugin requires %s or higher.<br>\', \'podlove-subscribe-button\' ), \'<code>PHP 5.3</code>\' );\n\techo \'<br />\';\n\tprintf( __( \'You are running %s\', \'podlove-subscribe-button\' ), \'<code>PHP \' . phpversion() . \'</code>\' );\n\texit;\n}'},
 {'size': 389,
  'snippet': "// Constants\nrequire('constants.php');\nrequire('settings/buttons.php');\n// Models\nrequire('model/base.php');\nrequire('model/button.php');\nrequire('model/network_button.php');\n// Table\nrequire('settings/buttons_list_table.php');\n// Media Types\nrequire('media_types.php');\n// Widget\nrequire('widget.php');\n// Version control\nrequire('version.php');\n// Helper functions\nrequire('helper.php');"},
 {'size': 1175,
  'snippet': "add_action( 'admin_menu', array( 'PodloveSubscribeButton', 'admin_menu') );\nif ( is_multisite() )\n\tadd_a

In [105]:
file = make_file_with_padding(FILE_ID, 30000, 0)
print(len(file))
print(file)

15
45
Prepend chars: 0
Append chars: 29433
buggy length: 567
Prepend snippets: 0
Append snippets: 29521
30088
<?php

public function get_data($query)
{
    $where = "";

    if (isset($_GET['group_id']) && $_GET['group_id']) {
        $group_id = sanitize_text_field($_GET['group_id']);
        $where = "WHERE group_ID = {$group_id}";
    }

    if (isset($_GET['country_code']) && $_GET['country_code']) {
        $country_code = sanitize_text_field($_GET['country_code']);
        $where .= " AND mobile LIKE '{$country_code}%'";
    }
    $query = $this->db->prepare("SELECT * FROM {$this->tb_prefix}sms_subscribes {$where}");

    return $this->db->get_results($query);
}
/**
 * Enqueue a style.
 *
 * @param string $handle The style handle.
 * @param string $src The source URL of the style.
 * @param array $deps An array of style dependencies.
 * @param string $media The context which style needs to be loaded: all, print, or screen
 * @return void
 * @example Assets::style('admin', 'dist/a

In [100]:
range1 = range(500, 28000, 500)

for i in range1:
    file = make_file_with_padding(FILE_ID,28000, i )
    if(abs(len(file)-28000) > 300):
        print("*"*20)
    print(f"expected length: {28000} and got {len(file)}")

32
54
Prepend chars: 500
Append chars: 27028
buggy length: 472
Prepend snippets: 552
Append snippets: 27125
expected length: 28000 and got 28149
32
54
Prepend chars: 1000
Append chars: 26528
buggy length: 472
Prepend snippets: 1054
Append snippets: 26624
expected length: 28000 and got 28150
32
54
Prepend chars: 1500
Append chars: 26028
buggy length: 472
Prepend snippets: 1556
Append snippets: 26123
expected length: 28000 and got 28151
32
54
Prepend chars: 2000
Append chars: 25528
buggy length: 472
Prepend snippets: 2055
Append snippets: 25620
expected length: 28000 and got 28147
32
54
Prepend chars: 2500
Append chars: 25028
buggy length: 472
Prepend snippets: 2557
Append snippets: 25120
expected length: 28000 and got 28149
32
54
Prepend chars: 3000
Append chars: 24528
buggy length: 472
Prepend snippets: 3060
Append snippets: 24617
expected length: 28000 and got 28149
32
54
Prepend chars: 3500
Append chars: 24028
buggy length: 472
Prepend snippets: 3560
Append snippets: 24118
expected l

In [ ]:
# create files with 30'000 characters with different bug positions, each should have ist own ID where FILEID_CHARS_BUGPOSITION